In [1]:
# To check how sensitive is the models biomass production to individual rxn deletions
import cobra
model = cobra.io.read_sbml_model("Helicobacter_pylori.xml")

'' is not a valid SBML 'SId'.


In [2]:
from cobra.flux_analysis import single_reaction_deletion

reaction_del = single_reaction_deletion(model)

print(reaction_del.head())
print("\nTotal reactions tested:", len(reaction_del))

           ids     growth   status
0  {3HAD11M12}  20.235589  optimal
1      {PPND2}  20.235589  optimal
2    {PSSA160}  20.235589  optimal
3       {HSDy}  20.235589  optimal
4    {PGPPI15}  20.235589  optimal

Total reactions tested: 1025


In [5]:
print(reaction_del.columns)
reaction_del.head()

Index(['ids', 'growth', 'status'], dtype='object')


,ids,growth,status
0,{3HAD11M12},20.235589,optimal
1,{PPND2},20.235589,optimal
2,{PSSA160},20.235589,optimal
3,{HSDy},20.235589,optimal
4,{PGPPI15},20.235589,optimal


In [7]:
# calculating reaction essentiality
# growth fraction = (growth after del)/(wt growth)

In [11]:
wt_growth = model.optimize().objective_value

reaction_del["growth_fraction"] = (
    reaction_del["growth"] / wt_growth
)

print("Wild-type biomass:", wt_growth)

print("\nEssential reactions:")
essential = reaction_del[
    reaction_del["growth_fraction"] < 0.01
]

print("Number:", len(essential))

Wild-type biomass: 20.23558895922475

Essential reactions:
Number: 112


In [13]:
growth_reducing = reaction_del[
    (reaction_del["growth_fraction"] >= 0.01) &
    (reaction_del["growth_fraction"] < 0.95)
]

print("Growth-reducing reactions:", len(growth_reducing))

Growth-reducing reactions: 1


In [15]:
non_essential = reaction_del[
    reaction_del["growth_fraction"] >= 0.95
]

print("Non-essential reactions:", len(non_essential))

Non-essential reactions: 912


In [19]:
import pandas as pd
# essential rxns
essential_ids = essential["ids"].tolist()

essential_info = []

for item in essential_ids:
    rxn_id = list(item)[0]
    rxn = model.reactions.get_by_id(rxn_id)

    essential_info.append({
        "Reaction": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Subsystem": rxn.subsystem
    })

essential_df = pd.DataFrame(essential_info)

print(essential_df.to_string(index=False))

      Reaction                                                                                                              Name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [21]:
essential_df.head(20)

,Reaction,Name,Equation,Subsystem
0,PAPPT3,phospho-N-acetylmuramoyl-pentapeptide-transfer...,udcpp[c] + ugmda[c] --> uagmda[c] + ump[c],
1,EX_fe2(e),EX_fe2(u),fe2[e] <=>,
2,SHK3Dr,shikimate dehydrogenase,3dhsk[c] + h[c] + nadph[c] <=> nadp[c] + skm[c],
3,IPDPUPT,"di-trans,poly-cis-Decaprenyl-diphosphate:isope...",7.0 ipdp[c] + ttc_ggdp[c] --> 7.0 ppi[c] + udc...,
4,FMNAT,FMN Adenylyltransferase,atp[c] + fmn[c] + h[c] --> fad[c] + ppi[c],
5,DHDPS,dihydrodipicolinate synthase,aspsa[c] + pyr[c] --> 23dhdp[c] + 2.0 h2o[c] +...,
6,VALTA,Valine Transaminase,akg[c] + val_L[c] <=> 3mob[c] + glu_L[c],
7,DHORTS,Dihydroorotase,dhor_S[c] + h2o[c] <=> cbasp[c] + h[c],
8,PSCVT,3-phosphoshikimate 1-carboxyvinyltransferase,pep[c] + skm5p[c] <=> 3psme[c] + pi[c],
9,DDPA,3-deoxy-D-arabino-heptulosonate 7-phosphate sy...,e4p[c] + h2o[c] + pep[c] --> 2dda7p[c] + pi[c],


In [23]:
essential_df["Type"] = essential_df["Reaction"].apply(
    lambda x:
        "Exchange" if x.startswith("EX_") else
        "Demand" if x.startswith("DM_") else
        "Sink" if x.startswith("SK_") else
        "Biomass" if "biomass" in x.lower() else
        "Internal"
)

print(essential_df["Type"].value_counts())

Type
Internal    95
Exchange    16
Biomass      1
Name: count, dtype: int64


In [25]:
print(
    essential_df[
        essential_df["Type"] == "Internal"
    ].head(30).to_string(index=False)
)

Reaction                                                                                            Name                                                          Equation Subsystem     Type
  PAPPT3                    phospho-N-acetylmuramoyl-pentapeptide-transferase (meso-2,6-diaminopimelate)                        udcpp[c] + ugmda[c] --> uagmda[c] + ump[c]           Internal
  SHK3Dr                                                                         shikimate dehydrogenase                   3dhsk[c] + h[c] + nadph[c] <=> nadp[c] + skm[c]           Internal
 IPDPUPT                                di-trans,poly-cis-Decaprenyl-diphosphate:isopentenyl-diphosphate              7.0 ipdp[c] + ttc_ggdp[c] --> 7.0 ppi[c] + udcpdp[c]           Internal
   FMNAT                                                                         FMN Adenylyltransferase                        atp[c] + fmn[c] + h[c] --> fad[c] + ppi[c]           Internal
   DHDPS                                          

In [27]:
# internal rxns that are essential 
total_internal = sum(
    not (
        rxn.id.startswith(("EX_", "DM_", "SK_"))
    )
    for rxn in model.reactions
)

essential_internal = len(
    essential_df[essential_df["Type"] == "Internal"]
)

print("Total internal reactions:", total_internal)
print("Essential internal reactions:", essential_internal)

print(
    "Essential internal percentage:",
    round(essential_internal / total_internal * 100, 2),
    "%"
)

Total internal reactions: 912
Essential internal reactions: 95
Essential internal percentage: 10.42 %
